# 6.2.2 Parameter-Efficient Fine-Tuning (PEFT) — LoRA

Demonstrate LoRA: approximate a weight matrix W ≈ A @ B with rank r. Show reconstruction error and trainable parameter count reduction vs full fine-tuning.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

Path('output').mkdir(exist_ok=True)
rng = np.random.default_rng(42)
d_in, d_out = 768, 768
W = rng.standard_normal((d_out, d_in))

def lora_error(W, rank):
    U, S, Vt = np.linalg.svd(W, full_matrices=False)
    W_r = (U[:, :rank] * S[:rank]) @ Vt[:rank, :]
    return np.linalg.norm(W - W_r, 'fro') / np.linalg.norm(W, 'fro')

ranks = [1, 2, 4, 8, 16, 32, 64, 128]
errors = [lora_error(W, r) for r in ranks]
params = [r * (d_in + d_out) for r in ranks]
full_params = d_in * d_out

for r, e, p in zip(ranks, errors, params):
    print(f'rank={r:4d}: error={e:.4f}, params={p:,} ({100*(1-p/full_params):.1f}% reduction)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ranks, errors, 'o-', color='tomato', lw=2)
axes[0].set(xlabel='LoRA Rank r', ylabel='Relative Error', title='Reconstruction Error vs Rank')
axes[0].grid(True, alpha=0.3)

axes[1].bar(range(len(ranks)), [p/1e3 for p in params], color='steelblue', tick_label=[str(r) for r in ranks])
axes[1].axhline(full_params/1e3, color='red', linestyle='--', lw=2, label='Full FT')
axes[1].set(xlabel='LoRA Rank r', ylabel='Params (K)', title='Trainable Params vs Rank')
axes[1].legend(); axes[1].grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('output/peft_lora.png')
print('Saved peft_lora.png')